In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from google import genai
import requests


def weather_data_call():

    lat = 26.9124
    lon = 75.7873

    response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={
            "lat": lat,
            "lon": lon,
            "appid": "26557c4c4803dc6e9a973d42f83e0672",
            "units": "metric"
        }
    )

    return response.json()

weather_tool_schema = {
    "name": "get_current_weather",

    "description": (
        "Get the current weather for a given city. "
        "Use this when the user asks about current weather, temperature, "
        "or weather conditions in any location."
    ),

    "parameters": {
        "type": "object",

        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g. 'Jaipur', 'Mumbai', 'Delhi'"
            },

            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit. Default: celsius"
            }
        },

        "required": ["city"]
    }
}

print(weather_data_call())

def get_current_weather(city: str, unit: str = "celsius") -> dict:

    city_data = {
        "jaipur": {
            "temp": 41,
            "condition": "Sunny",
            "humidity": 22
        },
        "delhi": {
            "temp": 39,
            "condition": "Hazy",
            "humidity": 35
        },
        "mumbai": {
            "temp": 31,
            "condition": "Humid and Cloudy",
            "humidity": 85
        },
        "bengaluru": {
            "temp": 27,
            "condition": "Pleasant",
            "humidity": 60
        },
        "kolkata": {
            "temp": 33,
            "condition": "Thunderstorms",
            "humidity": 82
        },
        "patna": {
            "temp": 34,
            "condition": "Partly Cloudy",
            "humidity": 70
        },
        "chennai": {
            "temp": 36,
            "condition": "Hot and Sunny",
            "humidity": 74
        },
        "hyderabad": {
            "temp": 32,
            "condition": "Clear Sky",
            "humidity": 48
        },
        "pune": {
            "temp": 29,
            "condition": "Light Rain",
            "humidity": 65
        },
        "lucknow": {
            "temp": 35,
            "condition": "Dusty",
            "humidity": 40
        },
        "bhopal": {
            "temp": 30,
            "condition": "Scattered Clouds",
            "humidity": 55
        },
        "ahmedabad": {
            "temp": 38,
            "condition": "Dry Heat",
            "humidity": 25
        }
    }

    data = city_data.get(
        city.lower(),
        {
            "temp": 30,
            "condition": "Clear",
            "humidity": 50
        }
    )

    temp = (
        data["temp"]
        if unit == "celsius"
        else round(data["temp"] * 9 / 5 + 32, 1)
    )

    return {
        "city": city,
        "temperature": temp,
        "unit": unit,
        "condition": data["condition"],
        "humidity": data["humidity"],
        "source": "Simulated Weather Database"
    }



import json


def execute_tool_call(tool_name: str, tool_args: dict, tool_registry: dict) -> str:
    fn = tool_registry.get(tool_name)

    if fn is None:
        return f"Error: Tool '{tool_name}' not found in registry"

    try:
        result = fn(**tool_args)
        return json.dumps(result, indent=2)

    except TypeError as e:
        return f"Error: Bad arguments for '{tool_name}': {e}"

    except Exception as e:
        return f"Error: Tool execution failed: {e}"
    



import json

# Register available tools
tool_registry = {
    "get_current_weather": get_current_weather
}

# ---------------- SCHEMA ----------------
print("SCHEMA (what LLM sees):")
print(f"  Name       : {weather_tool_schema['name']}")
print(f"  Description: {weather_tool_schema['description'][:70]}...")
print(
    f"  Parameters : "
    f"{list(weather_tool_schema['parameters']['properties'].keys())}"
)


# ---------------- FUNCTION ----------------
print("\nFUNCTION (your Python code — LLM never sees this):")

result = get_current_weather("Jaipur", "celsius")

print(f"  Direct call: get_current_weather('Jaipur') -> {result}")


# ---------------- EXECUTOR ----------------
print("\nEXECUTOR (bridges LLM decision to actual call):")

# Simulating what the LLM would return
llm_decided_to_call = {
    "tool_name": "get_current_weather",
    "args": {
        "city": "Mumbai"
    }
}

exec_result = execute_tool_call(
    llm_decided_to_call["tool_name"],
    llm_decided_to_call["args"],
    tool_registry
)

print("  LLM decided: call get_current_weather(city='Mumbai')")
print(f"  Executor ran it → {exec_result}")

{'coord': {'lon': 75.7873, 'lat': 26.9124}, 'weather': [{'id': 802, 'main': 'Clouds', 'description': 'scattered clouds', 'icon': '03d'}], 'base': 'stations', 'main': {'temp': 35.64, 'feels_like': 37.19, 'temp_min': 35.64, 'temp_max': 35.64, 'pressure': 1005, 'humidity': 36, 'sea_level': 1005, 'grnd_level': 960}, 'visibility': 10000, 'wind': {'speed': 4.62, 'deg': 289, 'gust': 4.42}, 'clouds': {'all': 34}, 'dt': 1781847926, 'sys': {'country': 'IN', 'sunrise': 1781827389, 'sunset': 1781877186}, 'timezone': 19800, 'id': 8199371, 'name': 'Rambagh', 'cod': 200}
SCHEMA (what LLM sees):
  Name       : get_current_weather
  Description: Get the current weather for a given city. Use this when the user asks ...
  Parameters : ['city', 'unit']

FUNCTION (your Python code — LLM never sees this):
  Direct call: get_current_weather('Jaipur') -> {'city': 'Jaipur', 'temperature': 41, 'unit': 'celsius', 'condition': 'Sunny', 'humidity': 22, 'source': 'Simulated Weather Database'}

EXECUTOR (bridges LLM